# E3′ semente de treino 7 — Kaggle Notebooks (GPU)**O que este notebook faz.** Treina o BERTimbau (o "classificador forte", quefica FORA do laço de aprendizado ativo) em até 9 braços e mede cada um napopulação reservada. É o item nº 1 do parecer da banca: repetir o E3′ com umasemente de treino diferente da 42 para mostrar que o resultado não é sorte dainicialização.Cada braço é **um ajuste fino completo** do BERTimbau — por isso GPU éobrigatória. Em CPU isto levaria dias.| Braço | Treino | Responde ||---|---|---|| A | itens anotados pelo pipeline real, rótulos do **oráculo** | o que o pipeline entrega || B | mesmos itens de A, rótulos **gold** | custo do ruído do oráculo (A−B) || C | mesma quantidade, sorteados do pool, gold | valor da seleção (B−C) || D | pool inteiro (50.000), gold | a régua (teto do pool) || E, E20…E35 | prefixos de 15k…35k da trajetória de entropia do E6, gold | varredura de orçamento |**O que varia com a semente.** Só o treino: inicialização do cabeçalho declassificação, embaralhamento dentro de cada época, *dropout*, e o sorteio dobraço C. O particionamento pool/população e a amostra de avaliação ficam**fixos na semente de dados 42**, para que os braços continuem pareáveis entresementes.---## Antes de dar play — configuração do KaggleNo painel da direita (*Notebook options*):1. **Accelerator**: `GPU P100` (preferível) ou `GPU T4 x2`. O código usa **uma**   GPU, então a P100 costuma render mais aqui do que duas T4.2. **Internet**: **ligado**. É necessário para `pip install`, para o `git clone`   e para baixar os pesos do BERTimbau do Hugging Face.3. **Persistence**: pode deixar desligado — a persistência aqui vem do   *output* do notebook, não do diretório de sessão (ver a célula de retomada).E em *Add-ons → Secrets*, cadastre:| Secret | Para quê | Obrigatório? ||---|---|---|| `GITHUB_TOKEN` | clonar `GHDaru/activelearning` se o repositório for privado | só se privado |## Sobre a duração e as quedas de sessãoA sessão do Kaggle tem teto de execução (hoje 12 h em GPU) e a conta tem cotasemanal de GPU. A estimativa para os 9 braços é **1,5–2,5 h**, com folgaconfortável — mas **quedas não custam trabalho**: o `run_e3prime.py` pulaqualquer braço que já tenha o seu `.json` no diretório de saída. Basta rodar denovo com os resultados parciais no lugar (célula 5).

In [ ]:

# 1) A GPU está mesmo ligada? Se esta célula disser "SEM GPU", pare aqui:
#    Notebook options -> Accelerator -> GPU P100 (ou T4 x2).
import subprocess, sys

print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout
      or ">>> SEM GPU — ligue o acelerador antes de continuar. <<<")
print("python:", sys.version.split()[0])

In [ ]:

# 2) Código: clona o repositório do experimento.
#    Vai para /tmp (e NÃO para /kaggle/working) de propósito: só /kaggle/working
#    vira output do notebook, e nós queremos que o output tenha os resultados,
#    não uma cópia do repositório.
import os, subprocess
from pathlib import Path

REPO_DIR = Path("/tmp/activelearning")
BRANCH = "main"          # run_kaggle.sh reescreve esta linha ao empurrar o notebook

token = ""
try:                      # Add-ons -> Secrets -> GITHUB_TOKEN (só se o repo for privado)
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception as exc:
    print(f"(sem GITHUB_TOKEN nos Secrets: {type(exc).__name__}) — tentando clone público")

url = (f"https://{token}@github.com/GHDaru/activelearning.git" if token
       else "https://github.com/GHDaru/activelearning.git")

if not REPO_DIR.exists():
    r = subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, url, str(REPO_DIR)],
                       capture_output=True, text=True)
    # o token nunca é impresso: só o código de saída e a última linha do stderr
    print("clone rc:", r.returncode, "|", r.stderr.strip().splitlines()[-1:] or "")
    assert r.returncode == 0, "clone falhou — repo privado sem GITHUB_TOKEN? token expirado?"

os.chdir(REPO_DIR)
print("cwd:", os.getcwd())
print("commit:", subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                                capture_output=True, text=True).stdout.strip())
print("dataset.csv:", Path("data/dataset.csv").stat().st_size, "bytes")

In [ ]:

# 3) Dependências. A imagem do Kaggle já traz torch com CUDA; falta transformers.
%pip -q install transformers scikit-learn

import torch
print("torch:", torch.__version__, "| CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:

# 4) Cache de anotações do oráculo — decide QUAIS braços dá para rodar.
#
#    Os braços A, B e C dependem de
#    experiments/e5cycle/results/annotation_cache_nemotron.jsonl, que é o
#    registro do que o oráculo NIM respondeu no ciclo real. Esse arquivo é
#    IGNORADO pelo git (.gitignore: experiments/*/results/*.jsonl), então NÃO
#    vem no clone. Para rodar A/B/C, suba-o como Kaggle Dataset privado e
#    anexe-o em "Add Input" — esta célula o encontra sozinho em /kaggle/input.
#
#    Sem ele, o notebook roda mesmo assim os 6 braços que não dependem do
#    oráculo (D e a varredura E/E20/E25/E30/E35) em vez de falhar.
import glob, shutil
from pathlib import Path

CACHE = Path("experiments/e5cycle/results/annotation_cache_nemotron.jsonl")
TODOS_OS_BRACOS = ["A", "B", "C", "E", "D", "E20", "E25", "E30", "E35"]
SEM_ORACULO     = ["E", "D", "E20", "E25", "E30", "E35"]

if not CACHE.exists():
    achados = glob.glob("/kaggle/input/**/annotation_cache_nemotron.jsonl", recursive=True)
    if achados:
        CACHE.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(achados[0], CACHE)
        print("cache anexado de:", achados[0])

if CACHE.exists():
    n = sum(1 for _ in CACHE.open(encoding="utf-8"))
    ARMS = TODOS_OS_BRACOS
    print(f"cache do oráculo OK ({n} linhas) — rodando os 9 braços")
else:
    ARMS = SEM_ORACULO
    print("!" * 70)
    print("ATENÇÃO: annotation_cache_nemotron.jsonl AUSENTE.")
    print("Os braços A, B e C serão PULADOS — sem eles não há A−B (ruído do")
    print("oráculo) nem B−C (valor da seleção) para a semente 7.")
    print("Para rodar tudo: suba o cache como Kaggle Dataset e use 'Add Input'.")
    print("!" * 70)

print("braços desta execução:", ",".join(ARMS))

In [ ]:

# 5) Saída e RETOMADA.
#
#    OUT fica em /kaggle/working porque só esse diretório vira o output do
#    notebook — é assim que os resultados sobrevivem ao fim da sessão e é de lá
#    que o `kaggle kernels output` os baixa.
#
#    Retomada: se você anexar como input (a) o output de uma execução anterior
#    deste notebook ou (b) um dataset com os JSONs parciais, esta célula os
#    copia para OUT e o runner pula esses braços automaticamente.
import glob, os, shutil
from pathlib import Path

SEED = 7                                   # semente de TREINO (a de dados é fixa em 42)
OUT = Path("/kaggle/working/e3prime_results")
OUT.mkdir(parents=True, exist_ok=True)

# Duas fontes de retomada, nesta ordem:
#   (a) o próprio repositório — se uma execução anterior já commitou parciais
#       em experiments/e2e3/results/, o clone da célula 2 os trouxe junto;
#   (b) qualquer input anexado (output de execução anterior deste notebook).
fontes = (sorted(glob.glob(f"experiments/e2e3/results/e3prime_*_s{SEED}*.json"))
          + sorted(glob.glob(f"/kaggle/input/**/e3prime_*_s{SEED}*.json", recursive=True)))
recuperados = 0
for src in fontes:
    dst = OUT / os.path.basename(src)
    if not dst.exists():
        shutil.copy(src, dst)
        recuperados += 1

prontos = sorted(p.stem.split("_")[1] for p in OUT.glob(f"e3prime_*_s{SEED}.json")
                 if not p.stem.endswith("_pred"))
print(f"arquivos recuperados de execuções anteriores: {recuperados}")
print(f"braços já concluídos (serão pulados): {prontos or 'nenhum'}")
print(f"braços a treinar agora: {[a for a in ARMS if a not in prontos] or 'nenhum — já acabou'}")

In [ ]:

# 6) A execução. Comando canônico: o particionamento dos dados fica fixo em 42
#    de propósito; só a semente de TREINO muda.
#
#    Se a sessão cair no meio, não se perde braço concluído: salve o notebook
#    ("Save & Run All"), anexe o output da execução anterior como input e rode
#    de novo — a célula 5 recupera o que já ficou pronto.
import subprocess, sys

cmd = [sys.executable, "experiments/e2e3/run_e3prime.py",
       "--arms", ",".join(ARMS),
       "--epochs", "3",
       "--batch-size", "128",
       "--eval-limit", "0",
       "--seed", str(SEED),
       "--out-dir", str(OUT)]
print(" ".join(cmd), flush=True)

# stream da saída em tempo real: em execução longa, log mudo é log inútil
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
log = OUT / f"e3prime_s{SEED}.log"
with log.open("a", encoding="utf-8") as fh:
    for linha in proc.stdout:
        print(linha, end="", flush=True)
        fh.write(linha)
rc = proc.wait()
print(f"\n>>> run_e3prime.py terminou com código {rc}")
assert rc == 0, "execução falhou — veja o log acima; rode de novo para retomar"

In [ ]:

# 7) Conferência do que sai daqui: os números por braço e o critério da hipótese.
#    É este bloco que vai na mensagem de conclusão ao agente `principal`.
import json

resultados = {}
for p in sorted(OUT.glob(f"e3prime_*_s{SEED}.json")):
    if p.stem.endswith("_pred"):
        continue
    d = json.loads(p.read_text(encoding="utf-8"))
    resultados[d["arm"]] = d

print(f"== E3′ semente de treino {SEED} ==")
print(f"{'braço':>5} {'n_treino':>9} {'Macro F1':>9} {'acurácia':>9}  {'fit (min)':>9}")
for a, d in sorted(resultados.items(), key=lambda kv: kv[1]["n_train"]):
    print(f"{a:>5} {d['n_train']:>9} {d['macro_f1']:>9.4f} {d['accuracy']:>9.4f} "
          f"{d['fit_seconds'] / 60:>9.1f}")

if "A" in resultados and "D" in resultados:
    fa, fd = resultados["A"]["macro_f1"], resultados["D"]["macro_f1"]
    frac = resultados["A"]["n_train"] / resultados["D"]["n_train"]
    veredito = "SUSTENTADA" if fa >= 0.95 * fd else "NÃO sustentada"
    print(f"\nHIPÓTESE F1(A) >= 0,95 x F1(D): {fa:.4f} vs {0.95 * fd:.4f} "
          f"com {frac:.1%} dos rótulos -> {veredito}")
else:
    print("\nHipótese não avaliável nesta execução: faltam os braços A e/ou D "
          "(o braço A depende do cache do oráculo — ver célula 4).")

faltando = [a for a in ARMS if a not in resultados]
print(f"\nbraços faltando: {faltando or 'nenhum'}")
print(f"arquivos em {OUT}: {len(list(OUT.glob('*.json')))}")

---## Como pegar os resultadosOs arquivos ficam em `/kaggle/working/e3prime_results/`:`e3prime_<braço>_s7.json` (as métricas) e `e3prime_<braço>_s7_pred.json` (aspredições, necessárias para os testes pareados de McNemar).**Pela interface**: *Save Version → Save & Run All*, e ao terminar baixe ooutput pela aba *Output* do notebook.**Pela linha de comando** (mais confiável para execução longa; precisa do tokenda API do Kaggle):```bashexperiments/e2e3/kaggle/run_kaggle.sh          # empurra, acompanha, baixa e retoma sozinho```O script trata a queda de sessão como caso normal: baixa o parcial, commita osbraços prontos e empurra uma nova versão, que retoma de onde parou.## Onde os resultados devem ser commitados`experiments/e2e3/results/` no repositório `activelearning` — os `_s7.json` e os`_s7_pred.json` juntos. Sem as predições não dá para rodar o McNemar pareadoentre braços, que é o que sustenta as afirmações de significância no Cap. 5.